In [ ]:
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
from sklearn.metrics import roc_curve
from scipy.optimize import brentq
from scipy.interpolate import interp1d

import config
from model import get_model

# ============================================================================
# 1. CÁC HÀM TÍNH METRICS (Tích hợp từ logic của nhóm)
# ============================================================================
def compute_eer(labels, scores):
    fpr, tpr, thresholds = roc_curve(labels, scores, pos_label=1)
    eer = brentq(lambda x: 1. - x - interp1d(fpr, tpr)(x), 0., 1.)
    return eer

def compute_mindcf(labels, scores, p_target=0.05, c_miss=1, c_fa=1):
    """Tính toán MinDCF - Một chỉ số quan trọng trong code nhóm bạn"""
    fpr, tpr, thresholds = roc_curve(labels, scores, pos_label=1)
    fnr = 1 - tpr
    dcf = c_miss * fnr * p_target + c_fa * fpr * (1 - p_target)
    return np.min(dcf)

# ============================================================================
# 2. HELPER EXTRACTOR (Xử lý việc lấy embedding từ model)
# ============================================================================
def extract_emb(model, fbank, hand):
    """Trích xuất và chuẩn hóa embedding 512-dim"""
    with torch.no_grad():
        # Lấy đặc trưng theo Mode (1: FBank, 2: Handcrafted, 3: Hybrid)
        if config.MODE == 1:
            emb = model.main_branch(fbank.to(config.DEVICE).unsqueeze(0))
        elif config.MODE == 2:
            emb = model.aux_branch(hand.to(config.DEVICE).unsqueeze(0))
        else:
            m_emb = model.main_branch(fbank.to(config.DEVICE).unsqueeze(0))
            a_emb = model.aux_branch(hand.to(config.DEVICE).unsqueeze(0))
            emb = model.fusion(m_emb, a_emb)
        
        # Normalize p=2 như code của bạn nhóm bạn
        return F.normalize(emb, p=2, dim=1)

# ============================================================================
# 3. MAIN INFERENCE LOOP (Tuần tự & Dễ debug)
# ============================================================================
def run_evaluation():
    # --- CẤU HÌNH ---
    # Lưu ý: Thay đổi số lượng speakers khớp với lúc train của bạn
    NUM_SPEAKERS = 1000 
    CHECKPOINT_PATH = "best_eer_model.pth"
    CSV_PATH = "test_list_gt.csv"
    
    # Files đặc trưng đã trích xuất
    FBANK_DATA = "test_O_fbank.pt"
    HAND_DATA = "all_features_MFBE_Pitch.pt"

    # 1. Load Model
    model = get_model(num_speakers=NUM_SPEAKERS, device=config.DEVICE)
    ckpt = torch.load(CHECKPOINT_PATH, map_location=config.DEVICE)
    model.load_state_dict(ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt)
    model.eval()
    print(f"✅ Đã load Model Mode {config.MODE}")

    # 2. Load Dữ liệu đặc trưng
    print("⏳ Đang load đặc trưng vào bộ nhớ...")
    f_dict = torch.load(FBANK_DATA) if os.path.exists(FBANK_DATA) else {}
    h_dict = torch.load(HAND_DATA) if os.path.exists(HAND_DATA) else {}
    df = pd.read_csv(CSV_PATH, sep='\t', header=None, names=['label', 'path1', 'path2'])

    all_scores = []
    all_labels = []

    # 3. Chạy từng cặp (Sequential)
    print(f"🚀 Chạy inference cho {len(df)} cặp...")
    for _, row in tqdm(df.iterrows(), total=len(df)):
        # Xử lý Key lệch: FBank dùng basename, Handcrafted dùng relpath
        k_f1, k_f2 = os.path.basename(row['path1']), os.path.basename(row['path2'])
        k_h1, k_h2 = row['path1'], row['path2']

        # Kiểm tra sự tồn tại của data
        if k_f1 not in f_dict or k_h1 not in h_dict: continue

        # Trích xuất embedding
        emb1 = extract_emb(model, f_dict[k_f1], h_dict[k_h1])
        emb2 = extract_emb(model, f_dict[k_f2], h_dict[k_h2])

        # Tính Cosine Similarity (Vì đã normalize nên chỉ cần nhân vô hướng)
        score = torch.sum(emb1 * emb2).item()
        
        all_scores.append(score)
        all_labels.append(int(row['label']))

    # 4. Xuất kết quả
    eer = compute_eer(all_labels, all_scores)
    mindcf = compute_mindcf(all_labels, all_scores)

    print("\n" + "="*40)
    print(f"📊 KẾT QUẢ ĐÁNH GIÁ CUỐI CÙNG:")
    print(f"🔹 EER     : {eer*100:.2f}%")
    print(f"🔹 MinDCF  : {mindcf:.4f} (p_target=0.05)")
    print("="*40)

if __name__ == "__main__":
    run_evaluation()